**Install Required Libraries**

In [2]:
import sys
!{sys.executable} -m pip install flask pandas nltk transformers torch openpyxl

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------ --------------- 1.0/1.7 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 6.4 MB/s  0:00:00
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/11.5 MB 9.4 MB/s eta 0:00:02
   ------------- -------------------------- 3.9/11.5 MB 9.7 MB/s eta 0:00:01
   --------------------- ------------------ 6.3/11.5 MB 10.1 MB/s eta 0:00:01
   ------------------------ --------------- 7.1/11.5 MB 10.1 MB/s eta 0:00:01
   ----------------------------- ---------- 8.4/11.5 MB 8.0 MB/s eta 0:00:01
   ----------------------------------- ---- 10.2/11.5 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.5 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 11.5/11.5 MB 7.1 MB/s  0:00:01
   ---------------------------------------- 0.0/770.3 kB ? eta -:--:--
   ------------- ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Import Libraries & Initialize the AI Models**

In [3]:
import os
import nltk
from flask import Flask, render_template, request
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline

# Download the AI dictionary for sentiment tracking
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# Load the advanced emotion analysis model
print("Loading AI models... please wait a moment.")
emotion_classifier = pipeline(
    "text-classification", 
    model="j-hartmann/emotion-english-distilroberta-base", 
    top_k=1
)
print("AI Models loaded successfully!")

c:\Users\Sunbal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading AI models... please wait a moment.


c:\Users\Sunbal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sunbal\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 22452.30it

AI Models loaded successfully!


**Define the Analysis Functions**

In [4]:
def get_sentiment(text):
    if not text.strip():
        return "Neutral"
    scores = sia.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

def get_emotion(text):
    if not text.strip():
        return "neutral"
    try:
        result = emotion_classifier(text)[0]
        return result[0]['label']
    except Exception:
        return "unknown"

**Create the Webpage Layout (HTML)**

In [5]:
# Create the templates folder automatically
os.makedirs("templates", exist_ok=True)

# Define the HTML webpage structure
html_layout = """
<!DOCTYPE html>
<html>
<head>
    <title>Sentiment Analyzer App</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 50px; background-color: #f7f9fa; }
        .box { max-width: 500px; margin: auto; background: white; padding: 25px; border-radius: 8px; box-shadow: 0px 4px 10px rgba(0,0,0,0.1); }
        textarea { width: 100%; height: 100px; padding: 10px; box-sizing: border-box; font-size: 16px; border: 1px solid #ccc; border-radius: 4px; }
        button { width: 100%; background-color: #007bff; color: white; padding: 12px; border: none; font-size: 16px; border-radius: 4px; cursor: pointer; margin-top: 10px; }
        button:hover { background-color: #0056b3; }
        .result { margin-top: 20px; padding: 15px; border-radius: 4px; background-color: #e9ecef; border-left: 6px solid #6c757d; }
    </style>
</head>
<body>
    <div class="box">
        <h2>Sentiment & Emotion Web App</h2>
        <p>Type your text below to analyze it instantly in your browser:</p>
        <form method="POST">
            <textarea name="review_text" placeholder="Enter text here..." required>{{ text }}</textarea>
            <button type="submit">Analyze Sentiment</button>
        </form>
        
        {% if sentiment %}
        <div class="result">
            <h3>Results:</h3>
            <p><strong>Overall Sentiment:</strong> {{ sentiment }}</p>
            <p><strong>Detected Emotion:</strong> {{ emotion }}</p>
        </div>
        {% endif %}
    </div>
</body>
</html>
"""

# Save the file
with open("templates/index.html", "w") as f:
    f.write(html_layout)

print("Webpage template built successfully!")

Webpage template built successfully!


**Start the Server & Launch the Web App**

In [ ]:
app = Flask(__name__)

@app.route("/", methods=["GET", "POST"])
def index():
    sentiment = None
    emotion = None
    text = ""
    
    if request.method == "POST":
        text = request.form["review_text"]
        sentiment = get_sentiment(text)
        emotion = get_emotion(text)
        
    return render_template("index.html", sentiment=sentiment, emotion=emotion, text=text)

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [11/Jul/2026 23:02:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Jul/2026 23:02:22] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [11/Jul/2026 23:03:29] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [11/Jul/2026 23:04:32] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [11/Jul/2026 23:05:04] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Jul/2026 23:05:04] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [11/Jul/2026 23:05:34] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [11/Jul/2026 23:13:25] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Jul/2026 23:13:25] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [11/Jul/2026 23:13:41] "POST / HTTP/1.1" 200 -
